# External Exposure Paths (BigID × Sentinel Data Lake)

**Question:** Which sensitive assets are accessible to identities outside the corporate domain?
Looks for non-`@contoso.com` UPNs holding any permission on classified assets.

In [ ]:
# === Setup: connect to the Microsoft Sentinel data lake ===
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

data_provider = MicrosoftSentinelProvider(spark)

WORKSPACE = "<YOUR_SENTINEL_WORKSPACE_NAME>"
TABLE = "BigIDDSPMCatalog_CL"

# Pull last 30 days of BigID catalog rows
df = data_provider.read_table(TABLE, WORKSPACE)
df = df.filter(F.col("TimeGenerated") > F.expr("current_timestamp() - INTERVAL 30 DAYS"))
df.printSchema()
print("Row count:", df.count())


## Find external-party edges

In [ ]:
from pyspark.sql.functions import from_json, col, explode, array_union, coalesce, array, lower

perm_schema = "Read array<string>, Write array<string>, FullControl array<string>"

sens = (
    df.withColumn("perms", from_json(col("AssetPermissions"), perm_schema))
      .filter(
          (col("Classification").contains("PHI")) |
          (col("Classification").contains("GDPR")) |
          (col("Classification").contains("Restricted")) |
          (col("Classification").contains("Confidential"))
      )
      .select(
          "AssetID", "AssetSource", "Classification",
          array_union(
              coalesce(col("perms.Read"), array()),
              array_union(
                  coalesce(col("perms.Write"), array()),
                  coalesce(col("perms.FullControl"), array()),
              ),
          ).alias("Users"),
      )
)

edges = (
    sens.withColumn("User", explode(col("Users")))
        .filter(lower(col("User")).contains("@"))
        .filter(~lower(col("User")).contains("@contoso.com"))
        .select("User", "AssetID", "Classification", "AssetSource")
)
edges.show(50, truncate=False)


## Roll up — top external parties by exposure

In [ ]:
result = (
    edges.groupBy("User")
    .agg(
        F.countDistinct("AssetID").alias("ExposedAssets"),
        F.collect_set("Classification").alias("Classifications"),
        F.collect_set("AssetSource").alias("Sources"),
    )
    .orderBy(F.desc("ExposedAssets"))
    .limit(20)
)
result.show(truncate=False)
result = edges.join(result.select("User"), "User", "inner").select("User", "AssetID").limit(120)


## Visualize

In [ ]:
# === Visualize as a graph ===
import matplotlib.pyplot as plt
import networkx as nx

pdf = result.toPandas()
print(f"Edges to draw: {len(pdf)}")
display(pdf.head(50))

G = nx.DiGraph()
for _, row in pdf.iterrows():
    src = str(row.iloc[0])
    dst = str(row.iloc[1])
    G.add_edge(src, dst)

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42, k=0.6)
nx.draw_networkx_nodes(
    G, pos,
    nodelist=[n for n in G.nodes if n in pdf.iloc[:, 0].values],
    node_color="#1f77b4", node_size=900, alpha=0.85,
)
nx.draw_networkx_nodes(
    G, pos,
    nodelist=[n for n in G.nodes if n in pdf.iloc[:, 1].values and n not in pdf.iloc[:, 0].values],
    node_color="#d62728", node_size=900, alpha=0.85,
)
nx.draw_networkx_edges(G, pos, arrows=True, edge_color="#888", alpha=0.6, width=1.2)
nx.draw_networkx_labels(G, pos, font_size=8)
plt.title("External Exposure — Third Parties → BigID Sensitive Assets", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()
